# МОМО: лабораторная № 1

## Численные методы линейной алгебры в задаче наименьших квадратов

# 1.1 Цель работы

Исследовать влияние ранга и обусловленности матрицы данных на численное решение задачи наи-
меньших квадратов; сопоставить прямой, итерационный и спектральный способы решения; изучить
действие сокращения размерности и регуляризации.
Работа строится как единый вычислительный эксперимент. Все сравниваемые методы используют
одно разбиение данных и согласованный способ подготовки признаков.

# 1.2 Исходные данные

## Superconductors: предсказание критической температуры

**Задача регрессии:** по числовым характеристикам химического состава материала предсказать его критическую температуру `critical_temp` в кельвинах (К). При охлаждении ниже этой температуры материал переходит в сверхпроводящее состояние. Данные содержат сведения об известных сверхпроводниках.

Одна строка — запись о материале. Файл `train.csv` содержит полный набор: **21 263 наблюдения, 81 входной признак и 1 целевой столбец**, пропуски отсутствуют по описанию UCI. Признаки уже рассчитаны автором; новые признаки в этом ноутбуке не создаются.

| Переменная | Роль | Тип | Смысл |
|---|---|---|---|
| `critical_temp` | Целевая переменная `y` | Вещественная | Экспериментальная критическая температура, К |
| `number_of_elements` | Признак | Целочисленный | Количество различных химических элементов в составе |
| Остальные 80 столбцов | Признаки | Вещественные | По 10 характеристик для каждого из 8 свойств элементов |

Целевой столбец исключается из `X` и не используется при вычислении исходных признаков: они получены из состава и свойств элементов, а температура — из экспериментальных данных.

**Источники:** [Kaggle: Superconductors](https://www.kaggle.com/datasets/anlgrbz/super-conductors), [UCI: Superconductivty Data](https://archive.ics.uci.edu/dataset/464/superconductivty+data). Автор подготовленного набора — Kam Hamidieh; исходные сведения собраны из NIMS SuperCon. Очистка и расчёт признаков описаны в [статье автора, раздел 2](https://arxiv.org/html/1803.10260).

## Свойства элементов, на которых основаны признаки

Это свойства **химических элементов, входящих в материал**. Средняя плотность элементов, например, не является прямым измерением плотности самого материала.

| Суффикс в названии столбца | Свойство элемента | Единицы исходного свойства | Смысл |
|---|---|---|---|
| `atomic_mass` | Атомная масса | а.е.м. | Масса атома |
| `fie` | Энергия первой ионизации | кДж/моль | Энергия удаления первого электрона |
| `atomic_radius` | Атомный радиус | пм | Размер атома |
| `Density` | Плотность | кг/м³ | Масса на единицу объёма при стандартных условиях |
| `ElectronAffinity` | Сродство к электрону | кДж/моль | Энергетическая характеристика присоединения электрона |
| `FusionHeat` | Теплота плавления | кДж/моль | Энергия перехода из твёрдого состояния в жидкое |
| `ThermalConductivity` | Теплопроводность | Вт/(м·К) | Способность проводить тепло |
| `Valence` | Валентность | Безразмерная | Типичное число образуемых химических связей |

Единицы указаны для базовых свойств. Энтропия — безразмерная производная характеристика. При подготовке исходного набора автор добавил 1,5 к значениям сродства к электрону для корректного расчёта логарифмов. [Описание базовых свойств](https://arxiv.org/html/1803.10260#S2.SS1).


## Производные характеристики: как получены 80 признаков

Для каждого из восьми свойств уже рассчитаны следующие десять характеристик. Взвешенные версии учитывают доли элементов в химической формуле. Подробные определения, включая специальные формулы взвешенной энтропии и размаха, приведены в [разделе 2.3 статьи](https://arxiv.org/html/1803.10260#S2.SS3).

| Префикс | Характеристика | Что описывает | Пример столбца |
|---|---|---|---|
| `mean_` | Среднее арифметическое | Средний уровень свойства по различным элементам | `mean_atomic_mass` |
| `wtd_mean_` | Взвешенное среднее | Средний уровень с учётом долей элементов | `wtd_mean_atomic_mass` |
| `gmean_` | Среднее геометрическое | Мультипликативное усреднение значений | `gmean_atomic_mass` |
| `wtd_gmean_` | Взвешенное геометрическое среднее | Геометрическое усреднение с учётом долей | `wtd_gmean_atomic_mass` |
| `entropy_` | Энтропия | Равномерность нормированных значений свойства | `entropy_atomic_mass` |
| `wtd_entropy_` | Взвешенная энтропия | Равномерность с учётом долей элементов | `wtd_entropy_atomic_mass` |
| `range_` | Размах | Разность максимального и минимального значений | `range_atomic_mass` |
| `wtd_range_` | Взвешенный размах | Размах значений после умножения на доли элементов | `wtd_range_atomic_mass` |
| `std_` | Стандартное отклонение | Разброс значений вокруг среднего | `std_atomic_mass` |
| `wtd_std_` | Взвешенное стандартное отклонение | Разброс с учётом долей элементов | `wtd_std_atomic_mass` |

**Итого:** 8 свойств × 10 характеристик + `number_of_elements` = **81 признак**. Все перечисленные столбцы уже находятся в CSV, отдельно вычислять их не требуется.


# 1.3 Постановка задачи

Для обучающей матрицы $X \in \mathbb{R}^{n \times p}$ и вектора ответов $y \in \mathbb{R}^{n}$ рассматривается задача:

$$
\min_{\beta \in \mathbb{R}^{p}} \|X\beta - y\|_2^2.
$$

Соответствующие нормальные уравнения:

$$
A\beta = b, \qquad A = X^{T}X, \qquad b = X^{T}y.
$$

Основной показатель качества прогноза — **RMSE на тестовой части**:

$$
\operatorname{RMSE}
=
\sqrt{\frac{1}{n_{\text{test}}}
\sum_{i=1}^{n_{\text{test}}}(y_i - \hat{y}_i)^2}.
$$

# 1.4 Обязательная часть

## Установка зависимостей, загрузка датасета и деление данных

Запускайте ячейки сверху вниз в Jupyter, VS Code, Google Colab или Kaggle. Требуется **Python 3.10+** и доступ в интернет. Команда `%pip` устанавливает библиотеки в окружение текущего ядра.

In [1]:
%pip install -q "kagglehub>=1.0.3" pandas numpy


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
from IPython.display import display

DATASET_HANDLE = "anlgrbz/super-conductors/versions/1"
CSV_NAME = "train.csv"

csv_path = Path(kagglehub.dataset_download(DATASET_HANDLE, path=CSV_NAME))
print(f"CSV загружен: {csv_path}")


/Users/aleksandr/Documents/GitReps/MOMO_labs/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CSV загружен: /Users/aleksandr/.cache/kagglehub/datasets/anlgrbz/super-conductors/versions/1/train.csv


In [3]:
df = pd.read_csv(csv_path)

print(f"Наблюдений: {df.shape[0]:,}")
print(f"Столбцов: {df.shape[1]}")
display(df.head())


Наблюдений: 21,263
Столбцов: 82


,number_of_elements,mean_atomic_mass,wtd_mean_atomic_mass,gmean_atomic_mass,wtd_gmean_atomic_mass,entropy_atomic_mass,wtd_entropy_atomic_mass,range_atomic_mass,wtd_range_atomic_mass,std_atomic_mass,...,wtd_mean_Valence,gmean_Valence,wtd_gmean_Valence,entropy_Valence,wtd_entropy_Valence,range_Valence,wtd_range_Valence,std_Valence,wtd_std_Valence,critical_temp
0,4,88.944468,57.862692,66.361592,36.116612,1.181795,1.062396,122.90607,31.794921,51.968828,...,2.257143,2.213364,2.219783,1.368922,1.066221,1,1.085714,0.433013,0.437059,29.0
1,5,92.729214,58.518416,73.132787,36.396602,1.449309,1.057755,122.90607,36.161939,47.094633,...,2.257143,1.888175,2.210679,1.557113,1.047221,2,1.128571,0.632456,0.468606,26.0
2,4,88.944468,57.885242,66.361592,36.122509,1.181795,0.975980,122.90607,35.741099,51.968828,...,2.271429,2.213364,2.232679,1.368922,1.029175,1,1.114286,0.433013,0.444697,19.0
3,4,88.944468,57.873967,66.361592,36.119560,1.181795,1.022291,122.90607,33.768010,51.968828,...,2.264286,2.213364,2.226222,1.368922,1.048834,1,1.100000,0.433013,0.440952,22.0
4,4,88.944468,57.840143,66.361592,36.110716,1.181795,1.129224,122.90607,27.848743,51.968828,...,2.242857,2.213364,2.206963,1.368922,1.096052,1,1.057143,0.433013,0.428809,23.0


In [4]:
TARGET = "critical_temp"

assert TARGET in df.columns, f"В CSV отсутствует целевой столбец {TARGET!r}."
assert df.shape == (21263, 82), f"Неожиданный размер таблицы: {df.shape}"
assert df.select_dtypes(include="number").shape[1] == df.shape[1], "Есть нечисловые столбцы."
assert not df.isna().any().any(), "В данных обнаружены пропуски."

print(f"Признаков: {df.shape[1] - 1}")
print(f"Пропусков: {int(df.isna().sum().sum())}")
print(f"Целевая переменная: {TARGET}")


Признаков: 81
Пропусков: 0
Целевая переменная: critical_temp


In [5]:
SEED = 42
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15

X = df.drop(columns=TARGET)
y = df[TARGET]

rng = np.random.default_rng(SEED)
shuffled_indices = rng.permutation(df.index.to_numpy())
n_train = int(len(df) * TRAIN_FRACTION)
n_val = int(len(df) * VAL_FRACTION)

train_indices = shuffled_indices[:n_train]
val_indices = shuffled_indices[n_train:n_train + n_val]
test_indices = shuffled_indices[n_train + n_val:]

X_train, X_val, X_test = (X.loc[idx].copy() for idx in (train_indices, val_indices, test_indices))
y_train, y_val, y_test = (y.loc[idx].copy() for idx in (train_indices, val_indices, test_indices))


In [6]:
assert df.index.is_unique, "Для разбиения нужны уникальные индексы строк."
assert np.isclose(TRAIN_FRACTION + VAL_FRACTION + TEST_FRACTION, 1.0)
assert len(X_train) + len(X_val) + len(X_test) == len(df)
assert X_train.index.intersection(X_val.index).empty
assert X_train.index.intersection(X_test.index).empty
assert X_val.index.intersection(X_test.index).empty
assert set(X_train.index) | set(X_val.index) | set(X_test.index) == set(df.index)

for features, target in ((X_train, y_train), (X_val, y_val), (X_test, y_test)):
    assert features.index.equals(target.index), "Индексы признаков и цели не совпадают."
    assert TARGET not in features.columns, "Целевой столбец попал в признаки."

split_summary = pd.DataFrame({
    "Подвыборка": ["train", "val", "test"],
    "Наблюдений": [len(X_train), len(X_val), len(X_test)],
    "Признаков": [X_train.shape[1], X_val.shape[1], X_test.shape[1]],
})
split_summary["Доля, %"] = (split_summary["Наблюдений"] / len(df) * 100).round(2)

print(f"Seed: {SEED}")
display(split_summary)


Seed: 42


,Подвыборка,Наблюдений,Признаков,"Доля, %"
0,train,14884,81,70.0
1,val,3189,81,15.0
2,test,3190,81,15.0


Определим матрицы A и b для трейновой части

In [7]:
A_train = X_train.T @ X_train
b_train = X_train.T @ y_train

## 1.4.1 Исследование матрицы данных

## 1.4.2. Прямое решение нормальных уравнений


## 1.4.3. Итерационное решение

## 1.4.4. Решение на основе SVD